In [1]:
from google.colab import drive
drive.mount('/content/drive')
# then copy your csv to /content/

Mounted at /content/drive


#### Step 2 - Install dependencies and load data:

In [2]:
!pip install transformers torch vaderSentiment -q

import pandas as pd
import numpy as np
from tqdm import tqdm
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.6 MB/s eta 0:00:00


In [29]:
from pathlib import Path
# Google Drive paths
DRIVE_BASE = Path("/content/drive/MyDrive")

# Input: Original company documents
INPUT_DIR = DRIVE_BASE / "ESG Data" / "company_sentiment"
OUTPUT_DIR = DRIVE_BASE / "ESG Data" / "company_sentiment"

COMPANY = "amd"  # amd, nvidia, hp, dell


print(f"Loaded input and output directories")
print(f"")

Loaded input and output directories



In [30]:
df = pd.read_csv(str(INPUT_DIR) + "/" + COMPANY + "_clean.csv")
print(f"Loaded {len(df)} reviews")

Loaded 2511 reviews


In [31]:


print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU: Tesla T4


#### Step 3 - Model A: VADER:

In [32]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

vader_scores = []
for text in tqdm(df["review_text"], desc="VADER"):
    scores = analyzer.polarity_scores(str(text))
    vader_scores.append(scores["compound"])  # range [-1, 1]

df["vader_score"] = vader_scores

print("VADER score distribution:")
print(df["vader_score"].describe())
print(f"\nVADER by sentiment label:")
print(df.groupby("sentiment_label")["vader_score"].mean())

VADER: 100%|██████████| 2511/2511 [00:00<00:00, 5383.36it/s]

VADER score distribution:
count    2511.000000
mean        0.608733
std         0.397649
min        -0.961800
25%         0.493900
50%         0.764400
75%         0.872000
max         0.994400
Name: vader_score, dtype: float64

VADER by sentiment label:
sentiment_label
negative    0.083431
neutral     0.456279
positive    0.679489
Name: vader_score, dtype: float64


#### Step 4 - Model B: nlptown star predictor:

In [33]:
from transformers import pipeline

# Model B: predicts 1-5 stars directly
star_predictor = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=0,
    truncation=True,
    max_length=512
)

texts = df["review_text"].tolist()
star_results = []
batch_size = 32

for i in tqdm(range(0, len(texts), batch_size), desc="nlptown"):
    batch = texts[i:i+batch_size]
    star_results.extend(star_predictor(batch))

# Extract predicted stars (labels are "1 star" through "5 stars")
df["nlptown_stars"] = [int(r["label"].split()[0]) for r in star_results]
df["nlptown_confidence"] = [r["score"] for r in star_results]

print("Predicted star distribution:")
print(df["nlptown_stars"].value_counts().sort_index())
print(f"\nPredicted vs actual star mean by sentiment:")
print(df.groupby("sentiment_label")[["overall_stars", "nlptown_stars"]].mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

nlptown: 100%|██████████| 79/79 [00:25<00:00,  3.07it/s]

Predicted star distribution:
nlptown_stars
1      62
2     171
3     577
4    1312
5     389
Name: count, dtype: int64

Predicted vs actual star mean by sentiment:
                 overall_stars  nlptown_stars
sentiment_label                              
negative              1.547059       2.158824
neutral               3.000000       3.198830
positive              4.505253       3.935468


#### Step 5 - Model C: RoBERTa 3-class:

In [34]:
# Model C: pos/neu/neg with probabilities
roberta_clf = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=0,
    truncation=True,
    max_length=512,
    top_k=None  # return all class probabilities
)

roberta_results = []
for i in tqdm(range(0, len(texts), batch_size), desc="RoBERTa"):
    batch = texts[i:i+batch_size]
    roberta_results.extend(roberta_clf(batch))

# Convert to continuous score: P(positive) - P(negative)
def roberta_to_score(result):
    d = {item["label"]: item["score"] for item in result}
    return d.get("positive", 0) - d.get("negative", 0)

# Also extract the top label
def roberta_top_label(result):
    return max(result, key=lambda x: x["score"])["label"]

df["roberta_score"] = [roberta_to_score(r) for r in roberta_results]
df["roberta_label"] = [roberta_top_label(r) for r in roberta_results]

print("RoBERTa score distribution:")
print(df["roberta_score"].describe())
print(f"\nRoBERTa label distribution:")
print(df["roberta_label"].value_counts())
print(f"\nRoBERTa score by sentiment label:")
print(df.groupby("sentiment_label")["roberta_score"].mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
RoBERTa: 100%|██████████| 79/79 [00:26<00:00,  2.99it/s]

RoBERTa score distribution:
count    2511.000000
mean        0.610063
std         0.453303
min        -0.951821
25%         0.489788
50%         0.811358
75%         0.912505
max         0.979224
Name: roberta_score, dtype: float64

RoBERTa label distribution:
roberta_label
positive    2023
neutral      294
negative     194
Name: count, dtype: int64

RoBERTa score by sentiment label:
sentiment_label
negative   -0.256555
neutral     0.370333
positive    0.724777
Name: roberta_score, dtype: float64


#### Step 6 - Model D: Siebert:

In [35]:
# Model D: binary pos/neg, high accuracy
siebert_clf = pipeline(
    "sentiment-analysis",
    model="siebert/sentiment-roberta-large-english",
    device=0,
    truncation=True,
    max_length=512
)

siebert_results = []
for i in tqdm(range(0, len(texts), batch_size), desc="Siebert"):
    batch = texts[i:i+batch_size]
    siebert_results.extend(siebert_clf(batch))

# Convert to continuous: positive confidence stays positive, negative flips sign
def siebert_to_score(result):
    if result["label"] == "POSITIVE":
        return result["score"]
    else:
        return -result["score"]

df["siebert_score"] = [siebert_to_score(r) for r in siebert_results]
df["siebert_label"] = [r["label"] for r in siebert_results]

print("Siebert score distribution:")
print(df["siebert_score"].describe())
print(f"\nSiebert label distribution:")
print(df["siebert_label"].value_counts())
print(f"\nSiebert score by sentiment label:")
print(df.groupby("sentiment_label")["siebert_score"].mean())

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Siebert: 100%|██████████| 79/79 [00:51<00:00,  1.53it/s]

Siebert score distribution:
count    2511.000000
mean        0.777968
std         0.623495
min        -0.999512
25%         0.997906
50%         0.998708
75%         0.998828
max         0.998938
Name: siebert_score, dtype: float64

Siebert label distribution:
siebert_label
POSITIVE    2234
NEGATIVE     277
Name: count, dtype: int64

Siebert score by sentiment label:
sentiment_label
negative   -0.515912
neutral     0.465168
positive    0.941518
Name: siebert_score, dtype: float64


#### Step 7 - Quick validation overview:

In [36]:
from scipy.stats import spearmanr

print("=" * 50)
print("VALIDATION: Spearman correlation with actual stars")
print("=" * 50)

models = {
    "VADER": "vader_score",
    "nlptown (pred. stars)": "nlptown_stars",
    "RoBERTa": "roberta_score",
    "Siebert": "siebert_score",
}

for name, col in models.items():
    rho, pval = spearmanr(df["overall_stars"], df[col])
    print(f"{name:25s}  ρ = {rho:.3f}  (p = {pval:.3e})")

VALIDATION: Spearman correlation with actual stars
VADER                      ρ = 0.301  (p = 8.666e-54)
nlptown (pred. stars)      ρ = 0.502  (p = 9.637e-161)
RoBERTa                    ρ = 0.463  (p = 2.350e-133)
Siebert                    ρ = 0.496  (p = 2.320e-156)


#### Separate Pros and Cons:

In [38]:
# Score pros and cons separately with RoBERTa
pros_results = []
cons_results = []
pros_texts = df["pros"].fillna("").tolist()
cons_texts = df["cons"].fillna("").tolist()

for i in tqdm(range(0, len(pros_texts), batch_size), desc="RoBERTa pros"):
    pros_results.extend(roberta_clf(pros_texts[i:i+batch_size]))

for i in tqdm(range(0, len(cons_texts), batch_size), desc="RoBERTa cons"):
    cons_results.extend(roberta_clf(cons_texts[i:i+batch_size]))

df["roberta_pros_score"] = [roberta_to_score(r) for r in pros_results]
df["roberta_cons_score"] = [roberta_to_score(r) for r in cons_results]

# Check correlations
for col in ["roberta_pros_score", "roberta_cons_score"]:
    rho, pval = spearmanr(df["overall_stars"], df[col])
    print(f"{col:30s}  ρ = {rho:.3f}  (p = {pval:.2e})")

RoBERTa cons: 100%|██████████| 79/79 [00:26<00:00,  3.03it/s]

roberta_pros_score              ρ = 0.159  (p = 1.30e-15)
roberta_cons_score              ρ = 0.346  (p = 1.02e-71)


#### Step 8 - Save and download:


In [39]:
# Save to Google Drive
output_path = str(OUTPUT_DIR) + "/"+ COMPANY + "_scored.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} reviews with {len(df.columns)} columns to Google Drive")
print(f"Columns: {list(df.columns)}")

Saved 2511 reviews with 22 columns to Google Drive
Columns: ['overall_stars', 'date', 'year', 'quarter', 'headline', 'job_title', 'employment_status', 'location', 'pros', 'cons', 'review_text', 'sentiment_label', 'star_filter', 'vader_score', 'nlptown_stars', 'nlptown_confidence', 'roberta_score', 'roberta_label', 'siebert_score', 'siebert_label', 'roberta_pros_score', 'roberta_cons_score']


#### Step 9 - Spearman-Validationtable

In [40]:
from scipy.stats import spearmanr

validation = {}
for name, col in [("VADER", "vader_score"), ("nlptown", "nlptown_stars"),
                  ("RoBERTa", "roberta_score"), ("Siebert", "siebert_score")]:
    rho, pval = spearmanr(df["overall_stars"], df[col])
    validation[name] = {
        "spearman_rho": round(rho, 3),
        "p_value": "< 0.001" if pval < 0.001 else round(pval, 6)
    }

val_df = pd.DataFrame(validation).T
output_path_validation = str(OUTPUT_DIR) + "/"+ COMPANY + "_validation.csv"
val_df.to_csv(output_path_validation, index=False)
print(val_df)

        spearman_rho  p_value
VADER          0.301  < 0.001
nlptown        0.502  < 0.001
RoBERTa        0.463  < 0.001
Siebert        0.496  < 0.001
